In [ ]:
"""
REST API for univ_db (SQL Server).
Run: uvicorn main:app --host 0.0.0.0 --port 8000 --reload
Docs: http://localhost:8000/docs
"""

from typing import List, Optional

import pyodbc
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

CONN_STR = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=univ_db;"
    "Trusted_Connection=yes;"
    # For SQL login use instead:
    # "UID=sa;PWD=your_password;"
)

app = FastAPI(title="University API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


# ---------- DB helpers ----------

def get_conn():
    return pyodbc.connect(CONN_STR)


def fetch_all(sql: str, params: tuple = ()):
    """Run a SELECT and return rows as a list of dicts."""
    with get_conn() as conn:
        cursor = conn.cursor()
        cursor.execute(sql, params)
        cols = [c[0] for c in cursor.description]
        return [dict(zip(cols, row)) for row in cursor.fetchall()]


def fetch_one(sql: str, params: tuple = ()):
    rows = fetch_all(sql, params)
    return rows[0] if rows else None


def execute(sql: str, params: tuple = ()):
    """Run INSERT/UPDATE. Returns the first column of the first row if the query has OUTPUT."""
    try:
        with get_conn() as conn:
            cursor = conn.cursor()
            cursor.execute(sql, params)
            result = cursor.fetchone() if cursor.description else None
            conn.commit()
            return result[0] if result else None
    except pyodbc.Error as e:
        # Errors raised by triggers (e.g. trip is full) end up here
        raise HTTPException(status_code=400, detail=str(e.args[1] if len(e.args) > 1 else e))


# ---------- Request models ----------

class LoginRequest(BaseModel):
    email: str
    password: str


class ScanRequest(BaseModel):
    student_id: int
    qr_token: str


class OrderItem(BaseModel):
    menu_item_id: int
    quantity: int


class OrderRequest(BaseModel):
    student_id: int
    restaurant_id: int
    items: List[OrderItem]


class BookingRequest(BaseModel):
    student_id: int
    seats: int = 1


class PaymentRequest(BaseModel):
    student_fee_id: int
    amount: float


# ---------- 1) Users & academic ----------

@app.post("/login")
def login(body: LoginRequest):
    user = fetch_one(
        "SELECT user_id, full_name, email, role, password_hash FROM users WHERE email = ?",
        (body.email,),
    )
    # NOTE: plain comparison for the demo. Use bcrypt in a real system.
    if not user or user["password_hash"] != body.password:
        raise HTTPException(status_code=401, detail="Wrong email or password")
    user.pop("password_hash")
    return user


@app.get("/students")
def get_students():
    return fetch_all("SELECT * FROM students")


@app.get("/students/{student_id}")
def get_student(student_id: int):
    student = fetch_one("SELECT * FROM students WHERE student_id = ?", (student_id,))
    if not student:
        raise HTTPException(status_code=404, detail="Student not found")
    return student


@app.get("/professors")
def get_professors():
    return fetch_all("SELECT * FROM professors")


@app.get("/faculties")
def get_faculties():
    return fetch_all("SELECT * FROM faculties")


@app.get("/departments")
def get_departments(faculty_id: Optional[int] = None):
    if faculty_id:
        return fetch_all("SELECT * FROM departments WHERE faculty_id = ?", (faculty_id,))
    return fetch_all("SELECT * FROM departments")


@app.get("/courses")
def get_courses():
    return fetch_all("SELECT * FROM courses")


@app.get("/students/{student_id}/classes")
def get_student_classes(student_id: int):
    return fetch_all(
        """
        SELECT cl.class_id, co.course_id, co.course_name, cl.semester_id, cl.professor_id
        FROM enrollments e
        JOIN classes cl ON cl.class_id = e.class_id
        JOIN courses co ON co.course_id = cl.course_id
        WHERE e.student_id = ?
        """,
        (student_id,),
    )


# ---------- 2) Attendance ----------

@app.post("/attendance/scan")
def scan_qr(body: ScanRequest):
    session = fetch_one(
        "SELECT session_id FROM lecture_sessions WHERE qr_token = ?",
        (body.qr_token,),
    )
    if not session:
        raise HTTPException(status_code=404, detail="Invalid QR code")

    already = fetch_one(
        "SELECT 1 AS x FROM attendance WHERE session_id = ? AND student_id = ?",
        (session["session_id"], body.student_id),
    )
    if already:
        raise HTTPException(status_code=400, detail="Already checked in")

    execute(
        "INSERT INTO attendance (session_id, student_id, status) VALUES (?, ?, 'present')",
        (session["session_id"], body.student_id),
    )
    return {"message": "Attendance recorded"}


@app.get("/students/{student_id}/attendance")
def get_attendance(student_id: int):
    return fetch_all("SELECT * FROM v_student_attendance WHERE student_id = ?", (student_id,))


# ---------- 3) Announcements ----------

@app.get("/announcements")
def get_announcements(faculty_id: Optional[int] = None):
    # NULL faculty_id means the announcement is for the whole university
    return fetch_all(
        "SELECT * FROM announcements WHERE faculty_id IS NULL OR faculty_id = ?",
        (faculty_id,),
    )


# ---------- 4) Restaurant ----------

@app.get("/restaurants")
def get_restaurants():
    return fetch_all("SELECT * FROM restaurants")


@app.get("/restaurants/{restaurant_id}/menu")
def get_menu(restaurant_id: int):
    return fetch_all("SELECT * FROM menu_items WHERE restaurant_id = ?", (restaurant_id,))


@app.post("/orders")
def create_order(body: OrderRequest):
    if not body.items:
        raise HTTPException(status_code=400, detail="Order is empty")

    order_id = execute(
        "INSERT INTO orders (student_id, restaurant_id) OUTPUT INSERTED.order_id VALUES (?, ?)",
        (body.student_id, body.restaurant_id),
    )

    for item in body.items:
        # Price is copied at order time; the trigger updates the order total
        execute(
            """
            INSERT INTO order_items (order_id, menu_item_id, quantity, unit_price)
            SELECT ?, menu_item_id, ?, price FROM menu_items WHERE menu_item_id = ?
            """,
            (order_id, item.quantity, item.menu_item_id),
        )

    return fetch_one("SELECT * FROM orders WHERE order_id = ?", (order_id,))


@app.get("/students/{student_id}/orders")
def get_student_orders(student_id: int):
    return fetch_all("SELECT * FROM orders WHERE student_id = ?", (student_id,))


# ---------- 5) Trips ----------

@app.get("/trips")
def get_trips():
    return fetch_all("SELECT * FROM v_trip_availability")


@app.post("/trips/{trip_id}/book")
def book_trip(trip_id: int, body: BookingRequest):
    # The trigger rejects the booking if the trip is full
    execute(
        "INSERT INTO trip_bookings (trip_id, student_id, seats) VALUES (?, ?, ?)",
        (trip_id, body.student_id, body.seats),
    )
    return {"message": "Booking confirmed"}


# ---------- 6) Fees ----------

@app.get("/students/{student_id}/fees")
def get_fees(student_id: int):
    return fetch_all("SELECT * FROM v_fee_status WHERE student_id = ?", (student_id,))


@app.post("/payments")
def pay(body: PaymentRequest):
    if body.amount <= 0:
        raise HTTPException(status_code=400, detail="Amount must be positive")
    execute(
        "INSERT INTO payments (student_fee_id, amount) VALUES (?, ?)",
        (body.student_fee_id, body.amount),
    )
    return {"message": "Payment recorded"}